# GridTrackNet fine-tune / train on your labelled clips

**Before running:** in Google Drive create `MyDrive/tennis_finetune/` and copy your workspace into it:

```
MyDrive/tennis_finetune/
  videos/   rally7.mp4 ...            (30 or 60 FPS)
  labels/   rally7_ball.csv ...        (from label_tool.py)
  models/   gridtracknet_weights_torch.npz   (starting weights; any .npz the tracker loads)
  code/     train_gridtracknet.py + tennis_tracker/gridtracknet.py (this notebook copies them if missing)
```

Runtime → Change runtime type → **GPU** (T4 is fine; L4/A100 faster).

Labels are consumed at 30 FPS cadence: every **2nd** frame of a 60 FPS clip, every frame of a 30 FPS clip — exactly what `pretrack.py`/`label_tool.py` produce.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/tennis_finetune')
for sub in ('videos', 'labels', 'models', 'code', 'runs'):
    (WORK / sub).mkdir(parents=True, exist_ok=True)
print('clips:', sorted(p.stem for p in (WORK / 'videos').iterdir()))
print('labels:', sorted(p.name for p in (WORK / 'labels').glob('*_ball.csv')))
print('models:', sorted(p.name for p in (WORK / 'models').glob('*.npz')))

In [ ]:
# Pull the trainer + model definition next to the data (copied from your repo's finetune/ and tennis_tracker/).
import shutil, sys
CODE = WORK / 'code'
(CODE / 'tennis_tracker').mkdir(exist_ok=True)
(CODE / 'tennis_tracker' / '__init__.py').touch()
missing = [p for p in (CODE / 'train_gridtracknet.py', CODE / 'tennis_tracker' / 'gridtracknet.py') if not p.is_file()]
if missing:
    print('Upload these files into MyDrive/tennis_finetune/code/ then re-run this cell:')
    for p in missing:
        print('  ', p.relative_to(WORK))
    raise SystemExit
sys.path.insert(0, str(CODE))
!pip -q install opencv-python-headless numpy
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
#@title Training settings
MODE = 'finetune'          #@param ['finetune', 'scratch']
START_WEIGHTS = 'gridtracknet_weights_torch.npz'  #@param {type:'string'}
VAL_CLIPS = ''              #@param {type:'string'}   # space-separated clip stems to hold out; empty = last clip
EPOCHS = 12                 #@param {type:'integer'}
BATCH_SIZE = 4              #@param {type:'integer'}
LR = 1.0                    #@param {type:'number'}   # Adadelta; 0.3 for a gentle fine-tune
TAG = 'run1'                #@param {type:'string'}

# Frames are cached on the fast local disk, not in Drive.
cmd = [sys.executable, str(CODE / 'train_gridtracknet.py'),
       '--videos', str(WORK / 'videos'), '--labels', str(WORK / 'labels'),
       '--weights', str(WORK / 'models' / START_WEIGHTS),
       '--save', str(WORK / 'models' / f'gridtracknet_{TAG}_best.npz'),
       '--data-dir', '/content/cache/data', '--run-dir', str(WORK / 'runs' / TAG),
       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--lr', str(LR), '--workers', '2']
if VAL_CLIPS.strip():
    cmd += ['--val-clips', *VAL_CLIPS.split()]
if MODE == 'scratch':
    cmd.append('--from-scratch')
print(' '.join(cmd))

In [ ]:
import subprocess
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()
print('exit code', process.returncode)

In [ ]:
# Loss curve + the winner decision (best.npz vs starting weights on the held-out clips).
import json, pandas as pd
run = WORK / 'runs' / TAG
print(pd.read_csv(run / 'results.csv').to_string(index=False))
print(json.dumps(json.loads((run / 'winner.json').read_text()), indent=2))
print('\nBest model saved as models/' + f'gridtracknet_{TAG}_best.npz' + ' — copy it into the repo as models/gridtracknet_weights_torch.npz to use it in the tracker.')

## Iterating

1. Copy the winning `.npz` to `models/gridtracknet_weights_torch.npz` in the repo (keep the old one renamed).
2. Back on the PC: `python finetune/pretrack.py --weights <new.npz>` on new videos → correct with `label_tool.py` → add the CSVs to Drive.
3. Re-run this notebook with `START_WEIGHTS` = the new winner. The trainer only replaces the saved best when the held-out recall goes up **without** more wrong-object detections, so a bad run cannot overwrite a good model.
4. `evaluate_archive.py --mode raw --archive <labels+videos dir>` on the PC gives the same recall / wrong / false-alarm numbers for any weights file.